> Notebook-friendly copy of `part-I/1.7-oop-for-environmental-systems-solutions.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [ ]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle do not ship every package this notebook imports.
# Colab and Kaggle start in an empty working directory.
# This is a no-op in an environment that is already set up.
import importlib.util
import subprocess
import sys

for module, package in {"pooch": "pooch"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

from pathlib import Path

Path("_files").mkdir(exist_ok=True)   # the folder this notebook writes into

# Solutions

**ℹ️ Reference solutions**

Worked solutions for the short exercises in [1.7-oop-for-environmental-systems-exercises.ipynb](1.7-oop-for-environmental-systems-exercises.ipynb).

## Exercise 1: A class with state and behaviour

Define a `Glacier` class with `name` and `area_km2` instance attributes, a `describe()` method returning a short string, and a `__repr__`. Create one and print both the object and its description.

In [ ]:
class Glacier:
    def __init__(self, name: str, area_km2: float):
        self.name = name
        self.area_km2 = area_km2

    def describe(self) -> str:
        return f"{self.name}: {self.area_km2} km^2"

    def __repr__(self) -> str:
        return f"Glacier({self.name!r}, {self.area_km2})"

g = Glacier("Aletsch", 78.0)
print(g)
print(g.describe())

## Exercise 2: Instance versus class attributes

Define a `Planet` class with a shared class attribute `g_earth = 9.81` and per-instance attributes `name` and `surface_gravity`. Create two planets and show that `g_earth` is shared while `surface_gravity` differs.

In [ ]:
class Planet:
    g_earth = 9.81                       # shared class attribute

    def __init__(self, name: str, surface_gravity: float):
        self.name = name
        self.surface_gravity = surface_gravity

earth = Planet("Earth", 9.81)
mars = Planet("Mars", 3.71)
print(Planet.g_earth, earth.g_earth, mars.g_earth)         # all 9.81
print(earth.surface_gravity, mars.surface_gravity)         # 9.81, 3.71

## Exercise 3: A dataclass record

Use `@dataclass` to define a `Reading` with `timestamp` (str) and `temp_celsius` (float). Show that two readings with equal fields compare equal, and print one to see the generated repr.

In [ ]:
from dataclasses import dataclass

@dataclass
class Reading:
    timestamp: str
    temp_celsius: float

print(Reading("2024-06-01", 18.2))
print(Reading("2024-06-01", 18.2) == Reading("2024-06-01", 18.2))

## Exercise 4: Read and mutate methods, with validation

Define a `Series` class holding a list of floats, with `add(x)` to append and `mean()` to return the average. `mean()` should raise `ValueError` if there are no values. Demonstrate both.

In [ ]:
class Series:
    def __init__(self):
        self.values: list[float] = []

    def add(self, x: float) -> None:
        self.values.append(x)

    def mean(self) -> float:
        if not self.values:
            raise ValueError("no values")
        return sum(self.values) / len(self.values)

s = Series()
s.add(1.0); s.add(3.0)
print(s.mean())

## Exercise 5: Composition

Define a `Catchment` class that holds several `Series` objects keyed by name (composition). Add an `add_series(name, series)` method and a `names()` method returning the keys. Build one with two series.

In [ ]:
class Catchment:
    def __init__(self, name: str):
        self.name = name
        self.series: dict[str, Series] = {}

    def add_series(self, name: str, series: "Series") -> None:
        self.series[name] = series

    def names(self) -> list[str]:
        return list(self.series)

c = Catchment("Aare")
c.add_series("temp", Series())
c.add_series("discharge", Series())
print(c.names())

## Exercise 6: Fix the shared class-level list

The class below shares one list across all instances. Rewrite it so each instance has its own `entries`, then show two loggers do not contaminate each other.

```python
class Logger:
    entries = []
    def __init__(self, name):
        self.name = name
    def log(self, msg):
        self.entries.append(msg)
```

In [ ]:
class Logger:
    def __init__(self, name: str) -> None:
        self.name = name
        self.entries: list[str] = []          # per-instance list

    def log(self, msg: str) -> None:
        self.entries.append(msg)

a = Logger("a"); b = Logger("b")
a.log("x"); b.log("y")
print(a.entries, b.entries)        # ['x'] ['y']

## Exercise 7: Inheritance and overriding

Define a base `Station` with a `name` attribute and a `kind()` method returning `"station"`. Define `RiverGauge(Station)` that overrides `kind()` to return `"river gauge"`. Show that a `RiverGauge` keeps the inherited name but reports the new kind.

In [ ]:
class Station:
    def __init__(self, name: str):
        self.name = name

    def kind(self) -> str:
        return "station"

class RiverGauge(Station):
    def kind(self) -> str:           # override
        return "river gauge"

g = RiverGauge("Aare")
print(g.name, g.kind())

## Exercise 8: An invariant with assert

Write `normalise(weights)` that divides each weight by the total, and asserts the invariant that the result sums to 1 (within a small tolerance). Test it on `[2.0, 2.0]`.

In [ ]:
def normalise(weights: list[float]) -> list[float]:
    total = sum(weights)
    out = []
    for w in weights:
        out.append(w / total)
    assert abs(sum(out) - 1.0) < 1e-9, "weights must sum to 1"
    return out

print(normalise([2.0, 2.0]))   # [0.5, 0.5]

## Exercise 9: A custom exception

Define `NegativeDischargeError(ValueError)` and a function `check(q_m3s)` that raises it when discharge is negative. Catch it and print the message.

In [ ]:
class NegativeDischargeError(ValueError):
    pass

def check(q_m3s: float) -> float:
    if q_m3s < 0.0:
        raise NegativeDischargeError(f"discharge {q_m3s} < 0")
    return q_m3s

try:
    check(-3.0)
except NegativeDischargeError as err:
    print("caught:", err)

## Exercise 10: Try / except / else / finally

Write `safe_divide(a, b)` that returns `a / b`, catches `ZeroDivisionError` (returning `None`), prints a message in the `else` branch when it succeeds, and always prints "done" in `finally`. Call it with `(6, 2)` and `(6, 0)`.

In [ ]:
def safe_divide(a: float, b: float) -> float | None:
    try:
        result = a / b
    except ZeroDivisionError:
        print("cannot divide by zero")
        return None
    else:
        print("division ok")
        return result
    finally:
        print("done")

print(safe_divide(6, 2))
print(safe_divide(6, 0))

## Exercise 11: Validate preconditions

Write `validate(temp_celsius, rh_percent)` that raises `ValueError` if the temperature is below absolute zero or the relative humidity is outside 0–100 %. Show it passing on valid input and raising on `rh_percent = 150`.

In [ ]:
def validate(temp_celsius: float, rh_percent: float) -> None:
    if temp_celsius < -273.15:
        raise ValueError("temperature below absolute zero")
    if not (0.0 <= rh_percent <= 100.0):
        raise ValueError("relative humidity out of range")

validate(20.0, 55.0)
try:
    validate(20.0, 150.0)
except ValueError as err:
    print("caught:", err)

## Exercise 12: Logging with levels

Configure logging to stdout at INFO level, get a logger, and emit a debug message (which should be suppressed), an info message, and a warning.

In [ ]:
import logging
import sys

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s",
                    stream=sys.stdout, force=True)
log: logging.Logger = logging.getLogger("demo")
log.debug("suppressed")
log.info("started")
log.warning("low sample count")

## Exercise 13: Write and run a pytest suite

Write a module `mymod.py` with `double(x)` returning `2 * x`, and a parametrised test file that checks three cases. Run pytest on it with a subprocess and print the output.

In [ ]:
from pathlib import Path
import subprocess
import sys

In [ ]:
Path("_files").mkdir(exist_ok=True)
Path("_files/mymod.py").write_text("def double(x: float) -> float:\n    return 2 * x\n", encoding="utf-8")
Path("_files/test_mymod.py").write_text(
    "import pytest\n"
    "from mymod import double\n"
    "@pytest.mark.parametrize('x, y', [(1, 2), (3, 6), (0, 0)])\n"
    "def test_double(x, y):\n"
    "    assert double(x) == y\n",
    encoding="utf-8",
)
result: subprocess.CompletedProcess = subprocess.run(
    [sys.executable, "-m", "pytest", "test_mymod.py", "-q"],
    capture_output=True, text=True, cwd="_files",
)
print(result.stdout.strip())

## Exercise 14: Replace assert-based validation

The line `assert rh_percent >= 0, "negative humidity"` disappears under `python -O`. Rewrite the check as a function that raises `ValueError`, so it fires regardless of optimisation. Demonstrate on a valid value.

In [ ]:
def validate_humidity(rh_percent: float) -> float:
    if rh_percent < 0.0:
        raise ValueError("negative humidity")
    return rh_percent

print(validate_humidity(55.0))

## Exercise 15: Validated earthquake events — a real dataset

The [USGS](https://earthquake.usgs.gov/) publishes a feed of every earthquake it records; this exercise reuses the archived one-month extract from 1.5's pandas exercises.

Combine both halves of this subchapter: a class that carries state, and exceptions that refuse to let it be built from bad data.

In [ ]:
# Pre-supplied: download and cache the earthquake data (same file used in 1.5).
import pooch

quakes_path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/usgs_earthquakes_2023_01.csv",
    known_hash="sha256:14e6100a41c55772f73a34daf1cd6ea38ba0fa49a86bb62a3930868614975df2",
    fname="usgs_earthquakes_2023_01.csv",
    path=pooch.os_cache("mlees"),
)

**Step 1.** Read the three columns the class needs.

In [ ]:
import pandas as pd

quakes = pd.read_csv(quakes_path)[["place", "mag", "depth"]]
print(quakes.shape)

**Step 2.** The class, with validation in the constructor.

In [ ]:
class EarthquakeEvent:
    """One recorded earthquake, validated at construction."""

    def __init__(self, place: str, magnitude: float, depth_km: float) -> None:
        if magnitude < 0.0 or depth_km < 0.0:
            raise ValueError(
                f"negative magnitude or depth: {magnitude}, {depth_km} km"
            )
        self.place = place
        self.magnitude = magnitude
        self.depth_km = depth_km

    def __repr__(self) -> str:
        return (f"EarthquakeEvent(place={self.place!r}, "
                f"magnitude={self.magnitude}, depth_km={self.depth_km})")

    def is_shallow(self, threshold_km: float = 70.0) -> bool:
        """Return whether the event is shallower than the threshold."""
        return self.depth_km < threshold_km

**Step 3.** Check that a valid event builds and an invalid one is refused.

In [ ]:
event = EarthquakeEvent("10 km NE of Somewhere", 4.2, 35.0)
print(event)
print("shallow?", event.is_shallow())

try:
    EarthquakeEvent("test", -1.0, 10.0)
except ValueError as err:
    print("refused, as it should be:", err)

**Step 4.** Build one event per row, letting bad rows fail individually.

In [ ]:
events = []
n_rejected = 0
for place, mag, depth in zip(quakes["place"], quakes["mag"], quakes["depth"]):
    try:
        events.append(EarthquakeEvent(place, mag, depth))
    except ValueError:
        n_rejected += 1

print(len(events), "built |", n_rejected, "rejected")

# nothing is rejected here: this month of the USGS feed happens to carry no negative
# magnitude or depth. The guard is still what makes the loop safe to run on a month
# that does -- step 3 is where it was actually exercised.

**Step 5.** Count the shallow events, and find the deepest with an explicit loop.

In [ ]:
n_shallow = 0
for event in events:
    if event.is_shallow():
        n_shallow += 1

deepest = None
for event in events:
    if deepest is None or event.depth_km > deepest.depth_km:
        deepest = event

print("shallow events:", n_shallow)
print("deepest:", deepest)

**Step 6.** The summary.

In [ ]:
print(len(events), "events,", n_rejected, "rejected,", n_shallow, "shallow")
print(deepest)

## Exercise 16: The solar system, continued

Exercise 17 of [1.2](1.2-data-structures-and-control-flow-exercises.ipynb) left the solar system
as a dictionary: `planet_masses_1e24kg`, mapping a name to a mass in units of 10^24 kg. A
dictionary keeps the name and the mass together only by convention — pass the mass somewhere on
its own and the name is gone, which is exactly what went wrong in that exercise's bonus question.

Now that classes are available, finish the job: give a planet a type, so that its name and its mass
travel as one object and the comparison you want to make becomes a method on it.

The dictionary is rebuilt below so this exercise stands on its own.

In [ ]:
# Pre-supplied: the dictionary from Exercise 17 of 1.2, rebuilt here.
planet_masses_1e24kg = {
    "mercury": 0.330,
    "venus": 4.87,
    "earth": 5.97,
    "mars": 0.642,
    "jupiter": 1898.0,
    "saturn": 568.0,
    "uranus": 86.8,
    "neptune": 102.0,
}

**Q12) Write a class `Planet` with two attributes, `name` and `mass_1e24kg`.**

Create one instance for Earth (5.97) and one for Jupiter (1898), then print each one's name and
mass to check they were stored.

Hint: give `__init__` type annotations, as 1.7's lecture does — `name: str`, `mass_1e24kg: float`.
They are not enforced at runtime, but they are the only place the unit is written down.

In [ ]:
class Planet:
    def __init__(self, name: str, mass_1e24kg: float) -> None:
        self.name = name
        self.mass_1e24kg = mass_1e24kg

earth = Planet("earth", planet_masses_1e24kg["earth"])
jupiter = Planet("jupiter", planet_masses_1e24kg["jupiter"])

print(earth.name, earth.mass_1e24kg)          # earth 5.97
print(jupiter.name, jupiter.mass_1e24kg)      # jupiter 1898.0

**Q13) Add a method `is_light` that compares a planet against Jupiter.**

It should report `True` when the planet is strictly lighter than Jupiter, `False` when it is
strictly heavier, and the string `"same mass!"` when the two are equal. Ask it whether Earth is
lighter than Jupiter, and whether Jupiter is lighter than itself.

In [ ]:
class Planet:
    def __init__(self, name: str, mass_1e24kg: float) -> None:
        self.name = name
        self.mass_1e24kg = mass_1e24kg

    def is_light(self):
        # as asked: prints rather than returns, and mixes bool with str
        if self.mass_1e24kg < planet_masses_1e24kg["jupiter"]:
            print(True)
        elif self.mass_1e24kg > planet_masses_1e24kg["jupiter"]:
            print(False)
        else:
            print("same mass!")

In [ ]:
Planet("earth", planet_masses_1e24kg["earth"]).is_light()        # True
Planet("jupiter", planet_masses_1e24kg["jupiter"]).is_light()    # same mass!

In [ ]:
# the method prints its answer and returns nothing, so the caller gets None
print(Planet("earth", planet_masses_1e24kg["earth"]).is_light())

**Q14) Now fix what Q13 asked you to build.**

The method you just wrote has two problems that 1.7's own material names directly.

1. It *prints* its answer instead of returning it, so a caller cannot use the result —
   `if earth.is_light():` will not do what you expect.
2. It returns a `bool` on two branches and a `str` on the third, so the caller has to test the
   type of the answer before trusting it.

Rewrite it as `compare_to_jupiter()` returning one of three strings — `"lighter"`, `"heavier"`,
`"same"` — and have the caller do the printing. Then loop over every planet in
`planet_masses_1e24kg`, build a `Planet` for each, and print the verdict for all eight.

In [ ]:
class Planet:
    def __init__(self, name: str, mass_1e24kg: float) -> None:
        self.name = name
        self.mass_1e24kg = mass_1e24kg

    def compare_to_jupiter(self) -> str:
        # one return type on every branch, and the caller decides what to do with it
        jupiter_mass_1e24kg = planet_masses_1e24kg["jupiter"]
        if self.mass_1e24kg < jupiter_mass_1e24kg:
            return "lighter"
        if self.mass_1e24kg > jupiter_mass_1e24kg:
            return "heavier"
        return "same"

In [ ]:
for name, mass_1e24kg in planet_masses_1e24kg.items():
    planet = Planet(name, mass_1e24kg)
    print(f"{planet.name:>8}: {planet.compare_to_jupiter()}")

**Q15) Give the class the invariant it is missing.**

A negative mass is not a planet. Raise a `ValueError` from `__init__` when `mass_1e24kg` is not
strictly positive, and show both that a valid planet still constructs and that
`Planet("nonsense", -1.0)` fails. Add a `__repr__` so a `Planet` prints readably.

This is the same shape as Exercise 15's `EarthquakeEvent`: validate at construction, so that every
object that exists is one you can trust.

In [ ]:
class Planet:
    def __init__(self, name: str, mass_1e24kg: float) -> None:
        if mass_1e24kg <= 0.0:
            raise ValueError(f"mass must be positive, got {mass_1e24kg}")
        self.name = name
        self.mass_1e24kg = mass_1e24kg

    def __repr__(self) -> str:
        return f"Planet(name={self.name!r}, mass_1e24kg={self.mass_1e24kg})"

    def compare_to_jupiter(self) -> str:
        jupiter_mass_1e24kg = planet_masses_1e24kg["jupiter"]
        if self.mass_1e24kg < jupiter_mass_1e24kg:
            return "lighter"
        if self.mass_1e24kg > jupiter_mass_1e24kg:
            return "heavier"
        return "same"

In [ ]:
print(Planet("earth", 5.97))        # Planet(name='earth', mass_1e24kg=5.97)

In [ ]:
try:
    Planet("nonsense", -1.0)
except ValueError as err:
    print("rejected:", err)         # rejected: mass must be positive, got -1.0

**ℹ️ What you just did**

You turned a dictionary into a type. The name and the mass are now one object rather than a key
and a value that happen to line up, the comparison is a method rather than a loose function that
has to be handed the right dictionary, and the constructor refuses to build a planet that could not
exist.

The Q13-to-Q14 step is worth a second look. Q13 asked for exactly what the exercise's earlier
edition asked for, and the result is a method that prints instead of returning and changes its
return type between branches. Both are easy to write and hard to use, and neither shows up until
something calls the method — which is why 1.7 spends its second half on making failures visible
early rather than on making code shorter.